#Build Constructor Standings

#### Sources
1. fact_session_results
1. dim_constructors

#### Output Columns
1. season
2. constructor Id
3. constructor name
4. nationality
5. race starts (number of races)
6. total points
7. number of wins
8. number of podiums
9. standing position (rank)

![formula1-gold-data-erd_1788542318941.png](./formula1-gold-data-erd_1788542318941.png "formula1-gold-data-erd_1788542318941.png")

In [0]:
%sql
CREATE OR REPLACE VIEW formula1.gold.v_constructor_standing
AS 
WITH constructor_session_summary AS (
    SELECT 
        fsr.season,
        dd.constructor_id,
        dd.constructor_name, 
        dd.nationality,
        COUNT(*) AS race_starts,
        SUM(fsr.points) AS total_points,
        count_if(fsr.is_win) AS number_wins,
        count_if(fsr.is_podium) AS number_podiums
    FROM formula1.gold.dim_constructors dd
    JOIN formula1.gold.fact_session_results fsr ON (dd.constructor_id = fsr.constructor_id) 
    GROUP BY dd.constructor_id, dd.constructor_name, dd.nationality, fsr.season
)

SELECT 
    *,
    RANK() OVER (PARTITION BY dss.season ORDER BY dss.total_points DESC, dss.number_wins DESC) as standing
FROM constructor_session_summary dss

In [0]:
%sql
SELECT * FROM formula1.gold.v_constructor_standing WHERE season = 2000